In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install ultralytics opencv-python -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 21.1 MB/s eta 0:00:00


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ Done! Video saved as output_counted.mp4
Final Count: 0


In [5]:
import os

base_path = "/content/drive/MyDrive"
print(os.listdir(base_path))

['Colab Notebooks', 'Getting started.pdf', 'JEE_MODULE 1_CHEM_ +1 NM Physical Chemistry 01.gdoc', 'anu-rvdc-vei - Oct 23, 2023.pdf', 'Sem 3 SPECIAL COMBO MID SEM (File responses)', 'Sem 1 SPECIAL COMBO MID SEM (File responses)', 'app.zip', 'codeVaikunth.pptx', 'val and val2 success and faliure .gdoc', 'Untitled document (4).gdoc', 'Screenshot_2025-08-18-16-35-07-95_c37d74246d9c81aa0bb824b57eaf7062.jpg', 'Screenshot 2025-08-29 105852.png', 'PATHWAY1.png', 'Screenshot 2025-08-29 110022.png', 'Document from Dhairya Saigal', 'FormSeven_copy (1).pdf', 'FormSeven_copy.pdf', 'ai-7YWNUF-dhairya-saigal-1757051152681-_copy.pdf', 'SIH_CODE_VAIKUNTH_2.pdf', 'sih level 1 result.xlsx', 'Guidelines for Internal Hackathon Participants.pdf', 'WhatsApp Image 2025-11-06 at 22.23.12_2d1ac679.jpg', 'IMG-20250921-WA0004(1).jpg', 'sib authorisation letter (1).docx', 'sib authorisation letter.docx', 'CODE VAIKUNTH PROTOTYPE', 'Screenshot 2025-09-29 205209.png', 'DHAIRYA_SAIGAL_RESUME update 2 (1).pdf', 'DHAIR

In [6]:
folder = "/content/drive/MyDrive/sack_prediction"

for f in os.listdir(folder):
    print(f)

Problem Statement Scenario1 (1).mp4
Problem Statement Scenario2.mp4
Problem Statement Scenario3.mp4
.ipynb_checkpoints


In [7]:
!pip install ultralytics opencv-python -q

In [8]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
print("Model loaded")

Model loaded


In [11]:
video_path = "/content/drive/MyDrive/sack_prediction/Problem Statement Scenario1 (1).mp4"

In [12]:
results = model.track(
    source=video_path,
    save=True,
    conf=0.25,
    tracker="bytetrack.yaml"
)

print("Tracking complete")


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/1994) /content/drive/MyDrive/sack_prediction/Problem Statement Scenario1 (1).mp4: 384x640 1 truck, 321.7ms
video 1/1 (frame 2/1994) /content/drive/MyDrive/sack_prediction/Problem Statement Scenario1 (1).mp4: 384x640 1 truck, 117.6ms
video 1/1 (frame 3/1994) /content/drive/MyDrive/sack_prediction/Problem Statement Scenario1 (1).mp4: 384x640 1 truck, 121.3ms
video 1/1 (frame 4/1994) /content/drive/MyDrive/sack_prediction/Problem Statemen

In [13]:
!ls runs/track

ls: cannot access 'runs/track': No such file or directory


In [14]:
from ultralytics import YOLO
import cv2
from collections import defaultdict

video_path = "/content/drive/MyDrive/sack_prediction/Problem Statement Scenario1 (1).mp4"

model = YOLO("yolov8n.pt")

cap = cv2.VideoCapture(video_path)

w = int(cap.get(3))
h = int(cap.get(4))
fps = int(cap.get(5))

out = cv2.VideoWriter(
    "/content/output_counted.mp4",
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,(w,h)
)

line_y = int(h*0.6)
seen_ids=set()
last_pos=defaultdict(int)
count=0

# 🔴 stream=True prevents RAM crash
for result in model.track(source=video_path, stream=True, persist=True, tracker="bytetrack.yaml"):

    frame = result.orig_img

    if result.boxes.id is not None:
        boxes = result.boxes.xyxy.cpu().numpy()
        ids = result.boxes.id.cpu().numpy()
        classes = result.boxes.cls.cpu().numpy()

        for box, track_id, cls in zip(boxes, ids, classes):

            # class 0 = person in COCO dataset
            if int(cls) != 0:
                continue

            x1,y1,x2,y2 = map(int, box)
            cy = (y1+y2)//2

            cv2.rectangle(frame,(x1,y1),(x2,y2),(0,255,0),2)
            cv2.putText(frame,f"Person {int(track_id)}",(x1,y1-10),
                        cv2.FONT_HERSHEY_SIMPLEX,0.6,(0,255,0),2)

            if last_pos[track_id] < line_y and cy >= line_y:
                if track_id not in seen_ids:
                    count+=1
                    seen_ids.add(track_id)

            last_pos[track_id]=cy

    cv2.line(frame,(0,line_y),(w,line_y),(0,0,255),2)
    cv2.putText(frame,f"COUNT:{count}",(30,50),
                cv2.FONT_HERSHEY_SIMPLEX,1,(0,0,255),3)

    out.write(frame)

out.release()
print("Done. Final count:",count)


video 1/1 (frame 1/1994) /content/drive/MyDrive/sack_prediction/Problem Statement Scenario1 (1).mp4: 384x640 1 truck, 99.9ms
video 1/1 (frame 2/1994) /content/drive/MyDrive/sack_prediction/Problem Statement Scenario1 (1).mp4: 384x640 1 truck, 106.9ms
video 1/1 (frame 3/1994) /content/drive/MyDrive/sack_prediction/Problem Statement Scenario1 (1).mp4: 384x640 1 truck, 107.9ms
video 1/1 (frame 4/1994) /content/drive/MyDrive/sack_prediction/Problem Statement Scenario1 (1).mp4: 384x640 1 truck, 99.9ms
video 1/1 (frame 5/1994) /content/drive/MyDrive/sack_prediction/Problem Statement Scenario1 (1).mp4: 384x640 1 truck, 102.9ms
video 1/1 (frame 6/1994) /content/drive/MyDrive/sack_prediction/Problem Statement Scenario1 (1).mp4: 384x640 1 person, 1 truck, 106.8ms
video 1/1 (frame 7/1994) /content/drive/MyDrive/sack_prediction/Problem Statement Scenario1 (1).mp4: 384x640 1 person, 1 truck, 101.3ms
video 1/1 (frame 8/1994) /content/drive/MyDrive/sack_prediction/Problem Statement Scenario1 (1).mp4

In [15]:
from google.colab import files
files.download("/content/output_counted.mp4")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [20]:
from ultralytics import YOLO
import cv2
from collections import defaultdict

video_path = "/content/drive/MyDrive/sack_prediction/Problem Statement Scenario1 (1).mp4"

model = YOLO("yolov8n.pt")

cap = cv2.VideoCapture(video_path)
w = int(cap.get(3))
h = int(cap.get(4))
fps = int(cap.get(5))

out = cv2.VideoWriter(
    "/content/output_video2.mp4",
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,(w,h)
)

line_y = int(h*0.65)

seen_ids=set()
last_pos=defaultdict(int)
count=0

truck_box=None

for result in model.track(source=video_path, stream=True, persist=True, tracker="bytetrack.yaml"):

    frame = result.orig_img

    if result.boxes.id is not None:
        boxes = result.boxes.xyxy.cpu().numpy()
        ids = result.boxes.id.cpu().numpy()
        classes = result.boxes.cls.cpu().numpy()

        # 🔹 First detect truck location
        for box, cls in zip(boxes, classes):
            if int(cls)==7:  # class 7 = truck
                truck_box = box

        for box, track_id, cls in zip(boxes, ids, classes):

            if int(cls)!=0:  # keep only persons
                continue

            x1,y1,x2,y2 = map(int, box)
            cy = (y1+y2)//2
            area=(x2-x1)*(y2-y1)

            # 🔹 FILTER 1: ignore very small detections
            if area < 5000:
                continue

            # 🔹 FILTER 2: must be near truck horizontally
            if truck_box is not None:
                tx1,ty1,tx2,ty2 = map(int,truck_box)
                if x2 < tx1-100 or x1 > tx2+100:
                    continue

            # Draw only filtered persons
            cv2.rectangle(frame,(x1,y1),(x2,y2),(0,255,0),2)
            cv2.putText(frame,f"Loader {int(track_id)}",(x1,y1-10),
                        cv2.FONT_HERSHEY_SIMPLEX,0.6,(0,255,0),2)

            # 🔹 Count crossing
            if last_pos[track_id] < line_y and cy >= line_y:
                if track_id not in seen_ids:
                    count+=1
                    seen_ids.add(track_id)

            last_pos[track_id]=cy

    # Draw counting line
    cv2.line(frame,(0,line_y),(w,line_y),(0,0,255),2)

    # Draw loading zone rectangle
    if truck_box is not None:
        tx1,ty1,tx2,ty2 = map(int,truck_box)
        cv2.rectangle(frame,(tx1-120,ty1-50),(tx2+120,ty2+80),(255,0,0),2)

    cv2.putText(frame,f"SACK COUNT:{count}",(30,50),
                cv2.FONT_HERSHEY_SIMPLEX,1,(0,0,255),3)

    out.write(frame)

out.release()
print("Done. Final count:",count)


video 1/1 (frame 1/1994) /content/drive/MyDrive/sack_prediction/Problem Statement Scenario1 (1).mp4: 384x640 1 truck, 116.9ms
video 1/1 (frame 2/1994) /content/drive/MyDrive/sack_prediction/Problem Statement Scenario1 (1).mp4: 384x640 1 truck, 106.4ms
video 1/1 (frame 3/1994) /content/drive/MyDrive/sack_prediction/Problem Statement Scenario1 (1).mp4: 384x640 1 truck, 110.0ms
video 1/1 (frame 4/1994) /content/drive/MyDrive/sack_prediction/Problem Statement Scenario1 (1).mp4: 384x640 1 truck, 105.3ms
video 1/1 (frame 5/1994) /content/drive/MyDrive/sack_prediction/Problem Statement Scenario1 (1).mp4: 384x640 1 truck, 102.3ms
video 1/1 (frame 6/1994) /content/drive/MyDrive/sack_prediction/Problem Statement Scenario1 (1).mp4: 384x640 1 person, 1 truck, 98.4ms
video 1/1 (frame 7/1994) /content/drive/MyDrive/sack_prediction/Problem Statement Scenario1 (1).mp4: 384x640 1 person, 1 truck, 103.8ms
video 1/1 (frame 8/1994) /content/drive/MyDrive/sack_prediction/Problem Statement Scenario1 (1).mp

In [21]:
from google.colab import files
files.download("/content/output_video2.mp4")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>